### Install Required Packages

In [1]:
!pip install openai faiss-cpu

  Using cached jiter-0.15.0-cp313-cp313-win_amd64.whl.metadata (5.3 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ------------------------------ --------- 1.0/1.4 MB 10.8 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 9.7 MB/s  0:00:00
Using cached jiter-0.15.0-cp313-cp313-win_amd64.whl (196 kB)
Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)

   -------------------------- ------------- 2/3 [openai]
   -------------------------- ------------- 2/3 [openai]
   -------------------------- ------------- 2/3 [openai]
   -------------------------- ------------- 2/3 [openai]
   -------------------------- ------------- 2/3 [openai]
   -------------------------- ------------- 2/3 [openai]
   -------------------------- ------------- 2/3 [openai]
   -------------------------- ------------- 2/3 [openai]
   -------------------------- ------------- 2/3 [openai]
   ---------------

###  Import Libraries

In [2]:
import pickle
import faiss
import numpy as np

### Load Metadata

In [3]:
with open("data/metadata.pkl", "rb") as f:

    metadata = pickle.load(f)

print("Metadata Loaded:", len(metadata))

Metadata Loaded: 6


### Load Vectorizer

In [4]:
with open("data/vectorizer.pkl", "rb") as f:

    vectorizer = pickle.load(f)

print("Vectorizer Loaded Successfully")

Vectorizer Loaded Successfully


### Load FAISS Index

In [5]:
index = faiss.read_index(
    "data/faiss_index.bin"
)

print("Vectors Stored:", index.ntotal)

Vectors Stored: 6


### Define Retrieval Function

In [6]:
def retrieve_context(question, k=3):

    query_vector = vectorizer.transform([question])

    query_vector = query_vector.toarray().astype("float32")

    distances, indices = index.search(
        query_vector,
        k
    )

    retrieved_chunks = []

    for idx in indices[0]:

        retrieved_chunks.append(
            {
                "source": metadata[idx]["source"],
                "text": metadata[idx]["text"]
            }
        )

    return retrieved_chunks

### Test Retrieval

In [7]:
question = "What is the password policy?"

results = retrieve_context(question)

### Display Retrieved Chunks

In [8]:
for result in results:

    print("Source:", result["source"])
    print(result["text"])
    print("-" * 80)

Source: cloud_migration_guide.txt
Cloud Migration Guide

AWS is the primary cloud platform.

Kubernetes is used for deployment.

GitHub Actions manages CI/CD.

Prometheus and Grafana are used for monitoring.
--------------------------------------------------------------------------------
Source: support_tickets.csv
TicketID           Issue Priority
0     T001   Login failure     High
1     T002       VPN issue   Medium
2     T003  Password reset      Low
--------------------------------------------------------------------------------
Source: company_policies.txt
Company Policies

Passwords must be changed every 90 days.

VPN access is mandatory.

Multi-factor authentication is required.

Confidential data should not be shared externally.
--------------------------------------------------------------------------------


### Build Context

In [9]:
def build_context(results):

    context = ""

    for item in results:

        context += item["text"]
        context += "\n\n"

    return context

### Generate Context

In [10]:
context = build_context(results)

print(context)

Cloud Migration Guide

AWS is the primary cloud platform.

Kubernetes is used for deployment.

GitHub Actions manages CI/CD.

Prometheus and Grafana are used for monitoring.

TicketID           Issue Priority
0     T001   Login failure     High
1     T002       VPN issue   Medium
2     T003  Password reset      Low

Company Policies

Passwords must be changed every 90 days.

VPN access is mandatory.

Multi-factor authentication is required.

Confidential data should not be shared externally.




### Create Prompt

In [11]:
prompt = f"""
Answer the question only using the provided context.

Context:
{context}

Question:
{question}

Answer:
"""

### View Prompt

In [12]:
print(prompt)


Answer the question only using the provided context.

Context:
Cloud Migration Guide

AWS is the primary cloud platform.

Kubernetes is used for deployment.

GitHub Actions manages CI/CD.

Prometheus and Grafana are used for monitoring.

TicketID           Issue Priority
0     T001   Login failure     High
1     T002       VPN issue   Medium
2     T003  Password reset      Low

Company Policies

Passwords must be changed every 90 days.

VPN access is mandatory.

Multi-factor authentication is required.

Confidential data should not be shared externally.



Question:
What is the password policy?

Answer:



### Create Simple Rule-Based Answer Function
Since I do not have an OpenAI API key yet, I'll first make the pipeline work.

In [19]:
def generate_answer(question):

    results = retrieve_context(question)

    answer = results[0]["text"]

    return {
        "question": question,
        "answer": answer,
        "sources": [results[0]["source"]]
    }

### Ask a Question

In [20]:
response = generate_answer(
    "What is the password policy?"
)

### Display Answer

In [21]:
print("Question:")
print(response["question"])

print("\nAnswer:")
print(response["answer"])

print("\nSources:")
print(response["sources"])

Question:
What is the password policy?

Answer:
Cloud Migration Guide

AWS is the primary cloud platform.

Kubernetes is used for deployment.

GitHub Actions manages CI/CD.

Prometheus and Grafana are used for monitoring.

Sources:
['cloud_migration_guide.txt']


### Test More Questions

In [23]:
questions = [
    "How many leave days are allowed?",
    "Who works in ML Engineering?",
    "What monitoring tools are used?"
]

for q in questions:

    result = generate_answer(q)

    print("=" * 80)
    print("Question:", q)
    print("Answer:", result["answer"])
    print("Sources:", result["sources"])

Question: How many leave days are allowed?
Answer: Employee Handbook

Employees receive 20 annual leave days.

Remote work is allowed three days per week.

Working hours are 9 AM to 6 PM.

Health insurance benefits are provided.
Sources: ['employee_handbook.txt']
Question: Who works in ML Engineering?
Answer: EmployeeID     Name      Department  Experience
0         101    Alice    Data Science           5
1         102      Bob  ML Engineering           3
2         103  Charlie              HR           8
3         104    David         Finance           6
4         105     Emma           Cloud           4
Sources: ['employees.csv']
Question: What monitoring tools are used?
Answer: Cloud Migration Guide

AWS is the primary cloud platform.

Kubernetes is used for deployment.

GitHub Actions manages CI/CD.

Prometheus and Grafana are used for monitoring.
Sources: ['cloud_migration_guide.txt']
